# Paper 4 – Experiments: CGRR vs baselines on the Kerala 2018 and 2019 floods

Runs everything in Sections 6–8 of the manuscript:

* **Main comparison** (Table 4): B1–B5 and CGRR, 200 Monte Carlo runs each
* **Ablations** A1 (fixed-width ranges), A2a/A2b (re-plan every period / never), A3 (no inundation or landslide routing)
* **Stress scenarios** S1–S3 (Table 5) and the **α × η sensitivity** grid
* **Calibration checks** (conformal coverage, set size, exceedance rate vs Proposition 2)
* Statistical tests: paired Wilcoxon signed-rank, Holm correction
* An **execution log** with timestamps, package versions and input-file checksums (proof of execution)

**How it works.** Plans use only what is known at decision time (forecasts and road closures reported in the previous period). So each method is planned once per event, then scored against 200 random draws of realized demand. This keeps the run time to minutes.

**Set `SMOKE_TEST = True` first** to check that everything runs on built-in random data. Those numbers are meaningless and must never go into the paper. Then set it to `False` and point the notebook to your real files.

In [ ]:
!pip -q install "pulp==2.9.0" highspy osmnx geopandas
# PuLP is pinned to 2.9: PuLP 4.x changed its API.

## 1. Settings

In [ ]:
SMOKE_TEST = True          # True = random test data (code check only). False = real data.
EVENTS     = ['2018', '2019']

from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/Paper4_Kerala'        # outputs of notebook 01 + your files
OUT  = DATA + ('/results_SMOKE_TEST' if SMOKE_TEST else '/results')
import os; os.makedirs(OUT, exist_ok=True)

### Model parameters
The defaults are **assumptions**. Every value you keep must be justified in Section 6 of the paper, with a citation or a data source.

* `kappa`: people one unit can serve per 6-hour period.
* `hmax`: the deepest water a resource can cross (m).
* `need_beta`: the share of exposed people who need rescue, evacuation or medical help in a period. Calibrate it from rescue and evacuation records. For 2018, NDRF alone rescued 535 people and evacuated 24,690; other agencies did much more.
* `delta_default`: used only if your calibration file has no `exposed_frac` column.

In [ ]:
import sys; sys.path.insert(0, '/content')
PARAMS = dict(
    alpha=0.10, eta=0.10, beta=0.5, tau_max=12.0, lam=0.01, mu=0.05, rho=0.5,
    fixed_width=0.30, b4_gamma_frac=0.5, n_runs=200, n_scen_b3=30, b5_search=60,
    need_beta=(2.0, 98.0),
    resources={
        'team':    {'kappa': 150, 'hmax': 0.3,  'mode': 'road'},
        'boat':    {'kappa': 60,  'hmax': 99.0, 'mode': 'boat'},
        'medical': {'kappa': 80,  'hmax': 0.3,  'mode': 'road'},
    },
    seed=42)

## 2. Pipeline code
These cells write the modules to disk. The same files are in the GitHub repository under `src/`.

In [ ]:
%%writefile /content/cgrr.py
"""CGRR experiment pipeline for Paper 4 (Natural Hazards submission).

Calibration-guided robust reallocation of flood emergency resources:
conformal sets -> demand ranges -> robust MIP (P) -> forecast-triggered
re-planning, evaluated in a hindcast digital twin against baselines
B1-B5 and ablations A1-A3.

Equation numbers refer to the manuscript (Sections 6 and 7).

Key design point: plans depend only on information available at
decision time (forecasts, reported road closures), never on the random
realization, so each method is planned ONCE per event and then scored
against all Monte Carlo realizations of demand.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd
import pulp
from scipy.stats import wilcoxon

CLASSES = ["low", "medium", "high", "severe"]
K = len(CLASSES)
INF = np.inf


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
@dataclass
class Config:
    alpha: float = 0.10            # conformal miscoverage level
    eta: float = 0.10              # trigger threshold, Eq. (trigger)
    beta: float = 0.5              # travel-time increase per metre of predicted depth
    tau_max: float = 12.0          # h, maximum useful travel time
    lam: float = 0.01              # weight of travel time in objective
    mu: float = 0.05               # weight of relocations in objective
    rho: float = 0.5               # equity floor share of lower demand
    M: float = 1000.0              # penalty of the soft equity constraint
    vuln_quantile: float = 0.75    # units with w above this quantile form I^v
    gamma_quantile: float = 0.95   # quantile used to calibrate Gamma
    fixed_width: float = 0.30      # +-30 % ranges for B4 and A1
    b4_gamma_frac: float = 0.5     # hand-tuned Gamma of B4 as share of units
    n_runs: int = 200
    n_scen_b3: int = 30
    b5_search: int = 60            # random-search candidates for B5
    # Beta(a,b): share of EXPOSED people needing rescue/evacuation/medical help
    # in a period (mean 0.02) -- ASSUMED; calibrate from rescue/evacuation records
    need_beta: tuple = (2.0, 98.0)
    # fraction of TOTAL unit population needing help, per risk class (used if
    # calibration data has no exposed_frac column) -- ASSUMED, justify in paper
    delta_default: tuple = (0.00001, 0.0001, 0.0005, 0.002)
    depth_proxy_m: float = 1.5     # depth proxy if forecasts lack pred_depth_m
    resources: dict = field(default_factory=lambda: {
        # kappa: people served per decision period; hmax: max crossable depth (m)
        "team":    {"kappa": 150, "hmax": 0.3, "mode": "road"},
        "boat":    {"kappa": 60,  "hmax": 99.0, "mode": "boat"},
        "medical": {"kappa": 80,  "hmax": 0.3, "mode": "road"},
    })
    solver_time: int = 30
    seed: int = 42


# ---------------------------------------------------------------------------
# Event container
# ---------------------------------------------------------------------------
@dataclass
class Event:
    name: str
    units: list
    bases: list
    res_names: list
    periods: list
    P: np.ndarray          # [I] total population
    w: np.ndarray          # [I] vulnerability weight
    v: np.ndarray          # [I] vulnerability multiplier
    probs: np.ndarray      # [T,I,K] forecast class probabilities
    true_cls: np.ndarray   # [T,I] observed class index
    pred_depth: np.ndarray # [T,I] predicted inundation depth (m)
    exposed: np.ndarray    # [T,I] observed exposed population
    tau0: np.ndarray       # [J,I] normal travel time (h)
    real_tt: dict          # mode -> [T,J,I] realized travel time, inf = impassable
    stocks: np.ndarray     # [R,J]

    @property
    def T(self): return len(self.periods)
    @property
    def I(self): return len(self.units)
    @property
    def J(self): return len(self.bases)
    @property
    def R(self): return len(self.res_names)

    def known_closed(self, t, mode):
        """Closures reported by the previous period (known at decision time)."""
        if t == 0:
            return np.zeros((self.J, self.I), bool)
        return ~np.isfinite(self.real_tt[mode][t - 1])


def load_event(name, forecasts, units, exposure, tau0, route_times, stocks, cfg,
               class_thresholds=(0.01, 0.05, 0.15)):
    """Build an Event from DataFrames (formats in the notebook / README).

    forecasts   : period, datetime, district, p_low..p_severe, [pred_depth_m], [true_class]
    units       : district, total_pop, [w], [v]
    exposure    : date, district, exposed_pop   (from notebook 01; forward-filled)
    tau0        : origin, district, tau0_hours
    route_times : date, origin, district, mode, tt_hours (blank/inf = impassable)
    stocks      : base, resource, count
    """
    f = forecasts.copy()
    f["datetime"] = pd.to_datetime(f["datetime"])
    per = f[["period", "datetime"]].drop_duplicates().sort_values("period")
    periods = list(per["datetime"])
    unit_list = list(units["district"])
    I, T = len(unit_list), len(periods)
    uidx = {u: i for i, u in enumerate(unit_list)}
    pidx = {p: t for t, p in enumerate(per["period"])}

    probs = np.zeros((T, I, K))
    depth = np.full((T, I), np.nan)
    true = np.full((T, I), -1)
    for r in f.itertuples(index=False):
        t, i = pidx[r.period], uidx[r.district]
        probs[t, i] = [getattr(r, f"p_{c}") for c in CLASSES]
        if "pred_depth_m" in f.columns:
            depth[t, i] = r.pred_depth_m
        if "true_class" in f.columns and isinstance(r.true_class, str):
            true[t, i] = CLASSES.index(r.true_class)
    probs = probs / probs.sum(-1, keepdims=True)
    if np.isnan(depth).any():
        print("NOTE: pred_depth_m missing -> depth proxy = "
              f"{cfg.depth_proxy_m} m x P(high or severe). State this in the paper.")
        proxy = cfg.depth_proxy_m * (probs[..., 2] + probs[..., 3])
        depth = np.where(np.isnan(depth), proxy, depth)

    # observed exposure, forward-filled from satellite dates to periods
    ex = exposure.copy()
    ex["date"] = pd.to_datetime(ex["date"])
    exposed = np.zeros((T, I))
    for i, u in enumerate(unit_list):
        s = ex[ex["district"] == u].sort_values("date").set_index("date")["exposed_pop"]
        if len(s):
            exposed[:, i] = s.reindex(periods, method="ffill").fillna(0).to_numpy()
    P = units["total_pop"].to_numpy(float)
    if (true < 0).any():
        frac = exposed / np.maximum(P, 1)[None, :]
        derived = np.digitize(frac, class_thresholds)
        print("NOTE: true_class missing for some rows -> derived from observed "
              f"exposed fraction with thresholds {class_thresholds}.")
        true = np.where(true < 0, derived, true)

    bases = list(dict.fromkeys(stocks["base"]))
    jidx = {b: j for j, b in enumerate(bases)}
    J = len(bases)
    tau = np.full((J, I), INF)
    for r in tau0.itertuples(index=False):
        if r.origin in jidx and r.district in uidx:
            tau[jidx[r.origin], uidx[r.district]] = r.tau0_hours

    rt = route_times.copy()
    rt["date"] = pd.to_datetime(rt["date"])
    rt["tt_hours"] = pd.to_numeric(rt["tt_hours"], errors="coerce").fillna(INF)
    modes = sorted({v["mode"] for v in cfg.resources.values()})
    real = {m: np.repeat(tau[None], T, 0).copy() for m in modes}
    for m in modes:
        sub = rt[rt["mode"] == m]
        for (o, d), g in sub.groupby(["origin", "district"]):
            if o not in jidx or d not in uidx:
                continue
            s = g.sort_values("date").set_index("date")["tt_hours"]
            s = s[~s.index.duplicated(keep="last")]
            vals = s.reindex(periods, method="ffill")
            vals = vals.fillna(tau[jidx[o], uidx[d]]).to_numpy()
            real[m][:, jidx[o], uidx[d]] = vals

    res_names = list(cfg.resources)
    st = np.zeros((len(res_names), J), int)
    for r in stocks.itertuples(index=False):
        st[res_names.index(r.resource), jidx[r.base]] += int(r.count)

    w = units["w"].to_numpy(float) if "w" in units else np.ones(I)
    v = units["v"].to_numpy(float) if "v" in units else np.ones(I)
    return Event(name, unit_list, bases, res_names, periods, P, w, v, probs, true,
                 depth, exposed, tau, real, st)


# ---------------------------------------------------------------------------
# Conformal prediction, damage fractions, Gamma (Sect. 6.1)
# ---------------------------------------------------------------------------
def fit_conformal(cal_probs, cal_true, alpha):
    """Split conformal, LAC score 1 - p_true. Returns threshold qhat."""
    n = len(cal_true)
    scores = 1.0 - cal_probs[np.arange(n), cal_true]
    level = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def conformal_sets(probs, qhat):
    S = (1.0 - probs) <= qhat
    top = probs.argmax(-1)
    np.put_along_axis(S, top[..., None], True, axis=-1)   # never empty
    return S


def estimate_delta(cal, cfg):
    d = np.array(cfg.delta_default, float)
    if "exposed_frac" in cal.columns:
        need_mean = cfg.need_beta[0] / sum(cfg.need_beta)
        g = (cal["exposed_frac"] * need_mean).groupby(cal["true_class"]).mean()
        for k, c in enumerate(CLASSES):
            if c in g.index and np.isfinite(g[c]):
                d[k] = g[c]
    return np.maximum.accumulate(d)          # monotone in class


def calibrate_gamma(cal, delta, q):
    probs = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    probs = probs / probs.sum(1, keepdims=True)
    true = cal["true_class"].map(CLASSES.index).to_numpy()
    exceed = delta[true] > probs @ delta
    counts = pd.Series(exceed).groupby(cal["period"].to_numpy()).sum()
    return int(math.ceil(np.quantile(counts, q)))


def bridge(probs, sets, delta, P, v):
    """Eq. (1): demand ranges from conformal sets."""
    vP = (v * P)[None, :]
    big = np.where(sets, delta[None, None, :], -INF).max(-1)
    small = np.where(sets, delta[None, None, :], INF).min(-1)
    return vP * small, vP * (probs @ delta), vP * big   # dL, dhat, dU


# ---------------------------------------------------------------------------
# Routes for planning (Eq. travel)
# ---------------------------------------------------------------------------
def planning_routes(ev, t, cfg, use_hazard=True, learned_closed=None):
    R, J, I = ev.R, ev.J, ev.I
    tau = np.zeros((R, J, I))
    allowed = np.ones((R, J, I), bool)
    for r, name in enumerate(ev.res_names):
        spec = cfg.resources[name]
        base = ev.tau0.copy()
        if use_hazard:
            h = np.broadcast_to(ev.pred_depth[t][None, :], (J, I))
            tau[r] = base * (1 + (cfg.beta * h if spec["mode"] == "road" else 0))
            allowed[r] &= h <= spec["hmax"]
            allowed[r] &= ~ev.known_closed(t, spec["mode"])
        else:
            tau[r] = base
            if learned_closed is not None:
                allowed[r] &= ~learned_closed[spec["mode"]]
        allowed[r] &= np.isfinite(tau[r]) & (tau[r] <= cfg.tau_max)
        if ev.stocks[r].sum() == 0:
            allowed[r] = False
        allowed[r] &= (ev.stocks[r] > 0)[:, None]
    return np.where(np.isfinite(tau), tau, cfg.tau_max * 10), allowed


# ---------------------------------------------------------------------------
# Optimization models
# ---------------------------------------------------------------------------
def _solver(cfg):
    return pulp.HiGHS(msg=False, timeLimit=cfg.solver_time)


def _xvars(allowed, stocks):
    idx = list(zip(*np.nonzero(allowed)))
    return {k: pulp.LpVariable("x_%d_%d_%d" % k, 0, int(stocks[k[0], k[1]]), cat="Integer")
            for k in idx}


def _extract(x, shape):
    out = np.zeros(shape, int)
    for k, var in x.items():
        out[k] = int(round(var.value() or 0))
    return out


def solve_robust(dhat, dU, dL, Gamma, tau, allowed, stocks, kappa, w, vul, xbar, cfg):
    """Model (P), Eqs. (3)-(9). Gamma = 0 and dU = dhat gives the nominal model."""
    R, J, I = allowed.shape
    m = pulp.LpProblem("P", pulp.LpMinimize)
    x = _xvars(allowed, stocks)
    u = [pulp.LpVariable(f"u{i}", 0) for i in range(I)]
    a = [pulp.LpVariable(f"a{i}", 0) for i in range(I)]
    pi = [pulp.LpVariable(f"pi{i}", 0) for i in range(I)]
    th = pulp.LpVariable("theta", 0)
    e = {i: pulp.LpVariable(f"e{i}", 0) for i in np.nonzero(vul)[0]}
    cov = [pulp.lpSum(kappa[r] * x[(r, j, i)] for r in range(R) for j in range(J)
                      if (r, j, i) in x) for i in range(I)]
    obj = (pulp.lpSum(w[i] * u[i] for i in range(I)) + Gamma * th + pulp.lpSum(pi)
           + cfg.lam * pulp.lpSum(tau[k] * x[k] for k in x)
           + cfg.M * pulp.lpSum(w[i] * e[i] for i in e))
    if xbar is not None:
        yp = {k: pulp.LpVariable("yp_%d_%d_%d" % k, 0) for k in x}
        ym = {k: pulp.LpVariable("ym_%d_%d_%d" % k, 0) for k in x}
        for k in x:
            m += x[k] - int(xbar[k]) == yp[k] - ym[k]
        obj += cfg.mu * pulp.lpSum(yp[k] + ym[k] for k in x)
    m += obj
    for i in range(I):
        m += u[i] >= dhat[i] - cov[i]
        m += a[i] >= dU[i] - cov[i]
        m += th + pi[i] >= w[i] * (a[i] - u[i])
    for r in range(R):
        for j in range(J):
            terms = [x[(r, j, i)] for i in range(I) if (r, j, i) in x]
            if terms:
                m += pulp.lpSum(terms) <= int(stocks[r, j])
    for i in e:
        m += cov[i] + e[i] >= cfg.rho * dL[i]
    m.solve(_solver(cfg))
    return _extract(x, (R, J, I))


def solve_stochastic(scen, tau, allowed, stocks, kappa, w, xbar, cfg):
    """B3: minimise expected weighted shortfall over sampled demand scenarios."""
    R, J, I = allowed.shape
    S = scen.shape[0]
    m = pulp.LpProblem("B3", pulp.LpMinimize)
    x = _xvars(allowed, stocks)
    z = [[pulp.LpVariable(f"z{s}_{i}", 0) for i in range(I)] for s in range(S)]
    cov = [pulp.lpSum(kappa[r] * x[(r, j, i)] for r in range(R) for j in range(J)
                      if (r, j, i) in x) for i in range(I)]
    obj = (pulp.lpSum(w[i] * z[s][i] for s in range(S) for i in range(I)) / S
           + cfg.lam * pulp.lpSum(tau[k] * x[k] for k in x))
    if xbar is not None:
        yp = {k: pulp.LpVariable("yp_%d_%d_%d" % k, 0) for k in x}
        ym = {k: pulp.LpVariable("ym_%d_%d_%d" % k, 0) for k in x}
        for k in x:
            m += x[k] - int(xbar[k]) == yp[k] - ym[k]
        obj += cfg.mu * pulp.lpSum(yp[k] + ym[k] for k in x)
    m += obj
    for s in range(S):
        for i in range(I):
            m += z[s][i] >= scen[s, i] - cov[i]
    for r in range(R):
        for j in range(J):
            terms = [x[(r, j, i)] for i in range(I) if (r, j, i) in x]
            if terms:
                m += pulp.lpSum(terms) <= int(stocks[r, j])
    m.solve(_solver(cfg))
    return _extract(x, (R, J, I))


def greedy(score, need, tau, allowed, stocks, kappa):
    """Assign resource units one at a time to the reachable unit with the best score.

    score(i_array, remaining, r, j) -> values for candidate units.
    """
    R, J, I = allowed.shape
    x = np.zeros((R, J, I), int)
    rem = need.astype(float).copy()
    for r in range(R):
        order = np.argsort(tau[r].min(1))
        for j in order:
            for _ in range(int(stocks[r, j])):
                cand = np.nonzero(allowed[r, j])[0]
                if len(cand) == 0:
                    break
                vals = score(cand, rem, r, j)
                i = cand[int(np.argmax(vals))]
                x[r, j, i] += 1
                rem[i] -= kappa[r]
    return x


# ---------------------------------------------------------------------------
# Policies
# ---------------------------------------------------------------------------
METHOD_INFO = {
    "B1": "Static deterministic allocation",
    "B2": "Proportional-to-risk heuristic",
    "B3": "Two-stage stochastic programming",
    "B4": "Rolling budgeted robust (fixed budget)",
    "B5": "Learned score-based dispatcher",
    "CGRR": "Proposed",
    "A1": "Fixed-width ranges",
    "A2a": "Re-plan every period",
    "A2b": "Never re-plan",
    "A3": "No inundation/landslide routing",
}


@dataclass
class Plan:
    x: np.ndarray          # [T,R,J,I]
    replans: np.ndarray    # [T] bool
    dU: np.ndarray         # [T,I] upper demand used for planning (for exceedance)
    reason: list           # [T] trigger reason or ''


def _sample_scenarios(probs_t, delta, P, v, S, rng):
    cum = probs_t.cumsum(-1)
    u = rng.random((S, probs_t.shape[0], 1))
    cls = (u > cum[None]).sum(-1).clip(max=K - 1)
    return (v * P)[None, :] * delta[cls]


def train_b5(ev, delta, qhat, cfg, rng):
    """B5: fit greedy score weights by random search on outcomes simulated
    from the forecasts only (never sees realized demand or roads)."""
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    best, best_val = None, INF
    cands = [np.array([1.0, 0.0, 0.0])] + [rng.uniform([0, 0, 0], [1, 1, 0.5])
                                            for _ in range(cfg.b5_search - 1)]
    T_train = list(range(0, ev.T, max(1, ev.T // 12)))
    sims = {t: _sample_scenarios(ev.probs[t], delta, ev.P, ev.v, 10, rng) for t in T_train}
    for th in cands:
        val = 0.0
        for t in T_train:
            tau, allowed = planning_routes(ev, t, cfg)
            _, dh, du = bridge(ev.probs[t:t + 1], conformal_sets(ev.probs[t:t + 1], qhat),
                               delta, ev.P, ev.v)
            dhat, spread = dh[0], np.maximum(du[0] - dh[0], 0)
            sc = lambda c, rem, r, j: (th[0] * ev.w[c] * np.maximum(rem[c], 0)
                                       + th[1] * ev.w[c] * np.maximum(spread[c], 0)
                                       - th[2] * tau[r, j, c] * ev.w[c] * dhat[c] / cfg.tau_max)
            x = greedy(sc, dhat, tau, allowed, ev.stocks, kappa)
            cov = np.einsum("r,rji->i", kappa, x)
            val += (ev.w[None] * np.maximum(sims[t] - cov[None], 0)).sum(1).mean()
        if val < best_val:
            best, best_val = th, val
    return best


def plan_method(ev, method, cfg, qhat, delta, Gamma, probs=None, b5_theta=None, rng=None):
    probs = ev.probs if probs is None else probs
    T, R, J, I = ev.T, ev.R, ev.J, ev.I
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    cap = float((kappa[:, None] * ev.stocks).sum())
    sets = conformal_sets(probs, qhat)
    dL, dhat, dU = bridge(probs, sets, delta, ev.P, ev.v)
    if method in ("B4", "A1"):
        dU = dhat * (1 + cfg.fixed_width)
        dL = dhat * (1 - cfg.fixed_width)
    G = {"B4": int(math.ceil(cfg.b4_gamma_frac * I))}.get(method, Gamma)
    vul = ev.w >= np.quantile(ev.w, cfg.vuln_quantile)
    if np.allclose(ev.w, ev.w[0]):
        vul = np.zeros(I, bool)
    # upper class used by the escalation trigger
    upper = sets.cumsum(-1).argmax(-1) if method != "A1" else probs.argmax(-1)

    xs = np.zeros((T, R, J, I), int)
    rep = np.zeros(T, bool)
    reasons = [""] * T
    xbar, ref = None, None
    learned = {m: np.zeros((J, I), bool) for m in ev.real_tt}
    every = method in ("B2", "B3", "B4", "B5", "A2a")
    for t in range(T):
        use_hazard = method != "A3"
        if method == "A3" and t > 0:          # learn closures only from own failures
            for r, name in enumerate(ev.res_names):
                mode = cfg.resources[name]["mode"]
                failed = (xs[t - 1, r] > 0) & ~np.isfinite(ev.real_tt[mode][t - 1])
                learned[mode] |= failed
        tau, allowed = planning_routes(ev, t, cfg, use_hazard,
                                       learned if method == "A3" else None)
        # ---- decide whether to (re)plan
        if xbar is None:
            fire, why = True, "initial"
        elif method in ("B1", "A2b"):
            fire, why = False, ""
        elif every:
            fire, why = True, "every period"
        else:
            d_ref = ref["dhat"]
            delta_t = (ev.w * np.abs(dhat[t] - d_ref)).sum() / max((ev.w * d_ref).sum(), cap)
            esc = bool((upper[t] > ref["upper"]).any())
            fail = bool(((xbar > 0) & ~allowed).any())
            fire = delta_t > cfg.eta or esc or fail
            why = ",".join(n for n, f in (("shift", delta_t > cfg.eta), ("escalation", esc),
                                          ("route", fail)) if f)
        if fire:
            if method == "B2":
                sc = lambda c, rem, r, j: ev.w[c] * np.maximum(rem[c], 0) - 1e-6 * tau[r, j, c]
                x = greedy(sc, dhat[t], tau, allowed, ev.stocks, kappa)
            elif method == "B5":
                th = b5_theta
                spread = np.maximum(dU[t] - dhat[t], 0)
                sc = lambda c, rem, r, j: (th[0] * ev.w[c] * np.maximum(rem[c], 0)
                                           + th[1] * ev.w[c] * spread[c]
                                           - th[2] * tau[r, j, c] * ev.w[c] * dhat[t][c] / cfg.tau_max)
                x = greedy(sc, dhat[t], tau, allowed, ev.stocks, kappa)
            elif method == "B3":
                scen = _sample_scenarios(probs[t], delta, ev.P, ev.v, cfg.n_scen_b3, rng)
                x = solve_stochastic(scen, tau, allowed, ev.stocks, kappa, ev.w, xbar, cfg)
            elif method == "B1":
                x = solve_robust(dhat[t], dhat[t], dL[t], 0, tau, allowed, ev.stocks,
                                 kappa, ev.w, vul, None, cfg)
            else:  # B4, CGRR, A1, A2a, A2b, A3
                x = solve_robust(dhat[t], dU[t], dL[t], G, tau, allowed, ev.stocks,
                                 kappa, ev.w, vul, xbar, cfg)
            xbar = x
            ref = {"dhat": dhat[t].copy(), "upper": upper[t].copy()}
            rep[t], reasons[t] = True, why
        xs[t] = xbar
    return Plan(xs, rep, dU, reasons)


# ---------------------------------------------------------------------------
# Hindcast evaluation (Sect. 7)
# ---------------------------------------------------------------------------
def draw_need(ev, cfg, n_runs, seed):
    rng = np.random.default_rng(seed)
    return rng.beta(*cfg.need_beta, size=(n_runs, ev.I))


def evaluate(ev, plan, cfg, need, demand_mult=1.0, sets=None):
    T, R, J, I = plan.x.shape
    N = need.shape[0]
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    d = (ev.v * ev.exposed)[None] * need[:, None, :] * demand_mult      # [N,T,I]
    cov = np.zeros((T, I))
    cov_plan = np.einsum("r,trji->ti", kappa, plan.x)
    rt = np.full((T, I), cfg.tau_max)
    failed = np.zeros(T)
    for r, name in enumerate(ev.res_names):
        tt = ev.real_tt[cfg.resources[name]["mode"]]                    # [T,J,I]
        ok = np.isfinite(tt) & (tt <= cfg.tau_max)
        xr = plan.x[:, r]
        cov += kappa[r] * (xr * ok).sum(1)
        failed += (xr * ~ok).sum((1, 2))
        cand = np.where((xr > 0) & ok, tt, INF).min(1)
        rt = np.minimum(rt, np.where(np.isfinite(cand), cand, cfg.tau_max))
    s = np.maximum(d - cov[None], 0)                                    # [N,T,I]
    D = d.sum((1, 2))
    out = {
        "worst_unit_weighted_unmet": (ev.w[None] * s.sum(1)).max(1),
        "total_weighted_unmet": (ev.w[None, None] * s).sum((1, 2)),
        "total_unmet_people": s.sum((1, 2)),
        "coverage_pct": 100 * (1 - s.sum((1, 2)) / np.maximum(D, 1e-9)),
        "mean_response_h": (d * rt[None]).sum((1, 2)) / np.maximum(D, 1e-9),
    }
    flat_rt = np.broadcast_to(rt[None], d.shape).reshape(N, -1)
    flat_d = d.reshape(N, -1)
    p95, w6 = np.zeros(N), np.zeros(N)
    order = np.argsort(flat_rt[0])
    for n in range(N):
        cw = np.cumsum(flat_d[n, order]) / max(flat_d[n].sum(), 1e-9)
        p95[n] = flat_rt[0, order][min(np.searchsorted(cw, 0.95), len(cw) - 1)]
        w6[n] = 100 * flat_d[n][flat_rt[0] <= 6].sum() / max(flat_d[n].sum(), 1e-9)
    out["p95_response_h"] = p95
    out["share_within_6h_pct"] = w6
    severe = (ev.true_cls == K - 1)[None]
    out["missed_high_risk"] = (severe & (s > 0.5 * d) & (d > 0)).sum((1, 2)).astype(float)
    reloc = np.maximum(np.diff(plan.x, axis=0), 0).sum() if T > 1 else 0
    out["relocations"] = np.full(N, float(reloc))
    out["replans"] = np.full(N, float(plan.replans.sum()))
    out["failed_dispatches"] = np.full(N, float(failed.sum()))
    a = np.maximum(plan.dU - cov_plan, 0)
    out["exceedance_rate"] = (s > a[None] + 1e-9).mean((1, 2))
    if sets is not None:
        hit = np.take_along_axis(sets, ev.true_cls[..., None], -1)[..., 0]
        out["set_coverage"] = np.full(N, hit.mean())
        out["mean_set_size"] = np.full(N, sets.sum(-1).mean())
    return pd.DataFrame(out)


METRICS_LOWER_BETTER = {
    "worst_unit_weighted_unmet": True, "total_weighted_unmet": True,
    "total_unmet_people": True, "coverage_pct": False, "mean_response_h": True,
    "p95_response_h": True, "share_within_6h_pct": False, "missed_high_risk": True,
    "relocations": True, "replans": True, "failed_dispatches": True,
}


def summarize(results, ref="CGRR"):
    """Median [IQR] per method, plus Wilcoxon signed-rank vs ref with Holm correction."""
    rows, tests = [], []
    for m, df in results.items():
        row = {"method": m}
        for c in METRICS_LOWER_BETTER:
            q1, med, q3 = np.percentile(df[c], [25, 50, 75])
            row[c] = med
            row[c + "_iqr"] = f"[{q1:.3g}, {q3:.3g}]"
        rows.append(row)
        if m == ref:
            continue
        for c in METRICS_LOWER_BETTER:
            diff = results[ref][c].to_numpy() - df[c].to_numpy()
            if np.allclose(diff, 0):
                p = 1.0
            elif np.allclose(diff, diff[0]):
                p = 0.0   # deterministic, constant non-zero difference across runs
            else:
                p = wilcoxon(results[ref][c], df[c]).pvalue
            tests.append({"method": m, "metric": c, "p": p,
                          "median_diff_ref_minus_method": float(np.median(diff))})
    tests = pd.DataFrame(tests)
    if len(tests):
        tests["p_holm"] = np.nan
        for c, g in tests.groupby("metric"):
            order = g["p"].sort_values().index
            k = len(order)
            adj, run = [], 0.0
            for rank, idx in enumerate(order):
                run = max(run, min(1.0, (k - rank) * tests.loc[idx, "p"]))
                adj.append((idx, run))
            for idx, val in adj:
                tests.loc[idx, "p_holm"] = val
    return pd.DataFrame(rows), tests


# ---------------------------------------------------------------------------
# Experiment driver
# ---------------------------------------------------------------------------
def degrade_probs(probs, true_cls):
    """Stress S3: double the forecast error vector (one-hot(true) - p)."""
    onehot = np.eye(K)[true_cls]
    p = np.clip(2 * probs - onehot, 1e-6, None)
    return p / p.sum(-1, keepdims=True)


def run_event(ev, cal, cfg, methods=None, log=print):
    methods = methods or list(METHOD_INFO)
    rng = np.random.default_rng(cfg.seed)
    cal_probs = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    cal_probs = cal_probs / cal_probs.sum(1, keepdims=True)
    cal_true = cal["true_class"].map(CLASSES.index).to_numpy()
    qhat = fit_conformal(cal_probs, cal_true, cfg.alpha)
    delta = estimate_delta(cal, cfg)
    Gamma = calibrate_gamma(cal, delta, cfg.gamma_quantile)
    log(f"[{ev.name}] qhat={qhat:.3f}  delta={np.round(delta, 5)}  Gamma={Gamma}")
    b5 = train_b5(ev, delta, qhat, cfg, rng) if "B5" in methods else None
    need = draw_need(ev, cfg, cfg.n_runs, cfg.seed)
    sets = conformal_sets(ev.probs, qhat)
    plans, res = {}, {}
    for m in methods:
        plans[m] = plan_method(ev, m, cfg, qhat, delta, Gamma, b5_theta=b5,
                               rng=np.random.default_rng(cfg.seed + 1))
        res[m] = evaluate(ev, plans[m], cfg, need, sets=sets)
        log(f"  {m:5s} replans={int(plans[m].replans.sum()):3d}  "
            f"coverage={res[m]['coverage_pct'].median():.1f}%")
    main, tests = summarize({m: res[m] for m in methods})
    info = {"qhat": qhat, "delta": delta, "Gamma": Gamma, "b5_theta": b5,
            "set_coverage": float(res[methods[0]]["set_coverage"].iloc[0]) if "set_coverage" in res[methods[0]] else None,
            "mean_set_size": float(sets.sum(-1).mean())}
    return {"plans": plans, "results": res, "summary": main, "tests": tests,
            "info": info, "need": need, "qhat": qhat, "delta": delta, "Gamma": Gamma, "b5": b5}


def run_stress(ev, cal, cfg, base, methods=("B1", "B2", "B3", "B4", "B5", "CGRR"), log=print):
    out = []
    qhat, delta, Gamma, b5, need = base["qhat"], base["delta"], base["Gamma"], base["b5"], base["need"]
    sets = conformal_sets(ev.probs, qhat)
    scen = {}
    # S1: demand 30 % higher than implied (realization only)
    scen["S1"] = {m: evaluate(ev, base["plans"][m], cfg, need, 1.3, sets) for m in methods}
    # S2: 25 % fewer resources
    ev2 = replace(ev, stocks=np.floor(ev.stocks * 0.75).astype(int))
    scen["S2"] = {m: evaluate(ev2, plan_method(ev2, m, cfg, qhat, delta, Gamma, b5_theta=b5,
                                               rng=np.random.default_rng(cfg.seed + 1)),
                              cfg, need, sets=sets) for m in methods}
    # S3: forecast errors doubled, no recalibration
    p3 = degrade_probs(ev.probs, ev.true_cls)
    ev3 = replace(ev, probs=p3)
    scen["S3"] = {m: evaluate(ev3, plan_method(ev3, m, cfg, qhat, delta, Gamma, b5_theta=b5,
                                               rng=np.random.default_rng(cfg.seed + 1)),
                              cfg, need, sets=conformal_sets(p3, qhat)) for m in methods}
    for s, res in scen.items():
        base_m = [m for m in methods if m != "CGRR"]
        best = min(base_m, key=lambda m: res[m]["total_weighted_unmet"].median())
        c, b = res["CGRR"], res[best]
        out.append({"scenario": s, "best_baseline": best,
                    "unmet_change_pct": 100 * (c["total_unmet_people"].median() / max(b["total_unmet_people"].median(), 1e-9) - 1),
                    "p95_change_pct": 100 * (c["p95_response_h"].median() / max(b["p95_response_h"].median(), 1e-9) - 1),
                    "coverage_change_pp": c["coverage_pct"].median() - b["coverage_pct"].median()})
        log(f"  {s}: best baseline {best}")
    return pd.DataFrame(out), scen


def run_sensitivity(ev, cal, cfg, alphas=(0.05, 0.10, 0.20, 0.30),
                    etas=(0.0, 0.05, 0.10, 0.20, 0.30), log=print):
    rows = []
    need = draw_need(ev, cfg, cfg.n_runs, cfg.seed)
    cal_probs = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    cal_probs = cal_probs / cal_probs.sum(1, keepdims=True)
    cal_true = cal["true_class"].map(CLASSES.index).to_numpy()
    delta = estimate_delta(cal, cfg)
    Gamma = calibrate_gamma(cal, delta, cfg.gamma_quantile)
    for a in alphas:
        qhat = fit_conformal(cal_probs, cal_true, a)
        sets = conformal_sets(ev.probs, qhat)
        for e in etas:
            c2 = replace(cfg, alpha=a, eta=e)
            p = plan_method(ev, "CGRR", c2, qhat, delta, Gamma)
            r = evaluate(ev, p, c2, need, sets=sets)
            rows.append({"alpha": a, "eta": e,
                         "set_coverage": r["set_coverage"].iloc[0],
                         "mean_set_size": r["mean_set_size"].iloc[0],
                         "total_weighted_unmet": r["total_weighted_unmet"].median(),
                         "worst_unit_weighted_unmet": r["worst_unit_weighted_unmet"].median(),
                         "coverage_pct": r["coverage_pct"].median(),
                         "relocations": r["relocations"].iloc[0],
                         "replans": r["replans"].iloc[0],
                         "exceedance_rate": r["exceedance_rate"].median()})
        log(f"  alpha={a} done")
    return pd.DataFrame(rows)


def latex_main_table(summary, methods):
    cols = ["worst_unit_weighted_unmet", "total_unmet_people", "mean_response_h",
            "p95_response_h", "coverage_pct", "missed_high_risk", "relocations"]
    names = {"B1": "B1 Static", "B2": "B2 Proportional", "B3": "B3 Stochastic",
             "B4": "B4 Fixed robust", "B5": "B5 Learned", "CGRR": "CGRR"}
    s = summary.set_index("method")
    lines = []
    for m in methods:
        vals = []
        for c in cols:
            v = s.loc[m, c]
            vals.append(f"{v:,.0f}" if abs(v) >= 100 else f"{v:.2f}")
        lines.append(f"{names.get(m, m)} & " + " & ".join(vals) + r" \\")
    return "\n".join(lines)


In [ ]:
%%writefile /content/routes.py
"""Build realized route travel times per date from the road graph,
flooded road segments (notebook 01) and landslide road blockages.

road mode : flooded and landslide-blocked segments are impassable
boat mode : landslide-blocked segments impassable; flooded segments
            passable at travel_time x boat_slowdown (boats on water)
"""
import numpy as np
import pandas as pd
import networkx as nx


def blocked_edges_from_landslides(edges_gdf, landslides_gdf, buffer_m=50):
    """edge_ids within buffer_m of any landslide point (both in any CRS)."""
    import geopandas as gpd
    e = edges_gdf.to_crs(32643)
    l = landslides_gdf.to_crs(32643)
    j = gpd.sjoin_nearest(l, e[["edge_id", "geometry"]], max_distance=buffer_m, how="inner")
    return set(j["edge_id"].astype(int))


def route_times(G, edge_table, origins, dests, dates, flooded_by_date, blocked_by_date,
                boat_slowdown=3.0, weight="travel_time"):
    """
    G               : networkx (Multi)DiGraph with `weight` in seconds
    edge_table      : DataFrame edge_id, u, v, key
    origins, dests  : dict name -> node
    dates           : list of date strings
    flooded_by_date : dict date -> set(edge_id)
    blocked_by_date : dict date -> set(edge_id)  (landslides active on that date)
    returns DataFrame date, origin, district, mode, tt_hours (inf = impassable)
    """
    et = edge_table.set_index("edge_id")
    Gu = G.to_undirected()
    base_w = {}
    for u, v, k, d in Gu.edges(keys=True, data=True):
        base_w[(u, v, k)] = float(d.get(weight, 1.0))

    def ekeys(ids):
        out = set()
        for e in ids:
            if e in et.index:
                r = et.loc[e]
                out.add((r.u, r.v, r.key))
                out.add((r.v, r.u, r.key))
        return out

    rows = []
    for date in dates:
        fl = ekeys(flooded_by_date.get(date, set()))
        bl = ekeys(blocked_by_date.get(date, set()))
        for mode in ("road", "boat"):
            H = nx.MultiGraph()
            H.add_nodes_from(Gu.nodes)
            for (u, v, k), w in base_w.items():
                if (u, v, k) in bl:
                    continue
                if (u, v, k) in fl:
                    if mode == "road":
                        continue
                    w = w * boat_slowdown
                H.add_edge(u, v, key=k, w=w)
            for o, on in origins.items():
                dist = nx.single_source_dijkstra_path_length(H, on, weight="w")
                for dname, dn in dests.items():
                    tt = dist.get(dn, np.inf)
                    rows.append({"date": date, "origin": o, "district": dname,
                                 "mode": mode, "tt_hours": tt / 3600.0})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /content/synthetic.py
"""SMOKE-TEST DATA ONLY. Generates random inputs in the exact formats the
pipeline expects, to check that the code runs. Never report these numbers."""
import numpy as np
import pandas as pd
from cgrr import CLASSES


def make(seed=0, I=14, J=5, T=40, n_cal_periods=60):
    rng = np.random.default_rng(seed)
    units = pd.DataFrame({"district": [f"D{i:02d}" for i in range(I)],
                          "total_pop": rng.integers(800_000, 3_500_000, I),
                          "w": rng.uniform(0.5, 1.5, I)})
    units["v"] = 1.0
    dates = pd.date_range("2018-08-08", periods=T, freq="6h")
    # latent severity rises then falls, differs by unit
    peak = rng.integers(10, 30, I)
    sev = np.array([[3.2 * np.exp(-((t - peak[i]) / 9.0) ** 2) for i in range(I)] for t in range(T)])
    noise = np.cumsum(rng.normal(0, 0.06, sev.shape), 0)      # slowly varying forecast error
    sev = sev + noise

    def probs_from(s):
        logits = -((np.arange(4)[None, None] - s[..., None]) ** 2)
        p = np.exp(logits); return p / p.sum(-1, keepdims=True)

    true = np.clip(np.round(sev + rng.normal(0, 0.5, sev.shape)), 0, 3).astype(int)
    pr = probs_from(sev)
    rows = []
    for t in range(T):
        for i in range(I):
            rows.append({"period": t, "datetime": dates[t], "district": units.district[i],
                         **{f"p_{c}": pr[t, i, k] for k, c in enumerate(CLASSES)},
                         "pred_depth_m": max(0, 0.6 * sev[t, i] + rng.normal(0, 0.1)),
                         "true_class": CLASSES[true[t, i]]})
    forecasts = pd.DataFrame(rows)
    # calibration periods (pre-event)
    sc = rng.uniform(0, 3.2, (n_cal_periods, I)) + rng.normal(0, 0.35, (n_cal_periods, I))
    tc = np.clip(np.round(sc + rng.normal(0, 0.5, sc.shape)), 0, 3).astype(int)
    pc = probs_from(sc)
    cal = pd.DataFrame([{"period": p, "district": units.district[i],
                         **{f"p_{c}": pc[p, i, k] for k, c in enumerate(CLASSES)},
                         "true_class": CLASSES[tc[p, i]],
                         "exposed_frac": [0.0005, 0.005, 0.02, 0.06][tc[p, i]] * rng.uniform(0.5, 1.5)}
                        for p in range(n_cal_periods) for i in range(I)])
    # observed exposure on satellite dates (every 4th period)
    frac = np.array([0.0005, 0.005, 0.02, 0.06])[true]
    exp_rows = [{"date": dates[t], "district": units.district[i],
                 "exposed_pop": units.total_pop[i] * frac[t, i] * rng.uniform(0.7, 1.3)}
                for t in range(0, T, 4) for i in range(I)]
    exposure = pd.DataFrame(exp_rows)
    bases = [f"B{j}" for j in range(J)]
    tau0 = pd.DataFrame([{"origin": b, "district": u, "tau0_hours": rng.uniform(0.5, 6)}
                         for b in bases for u in units.district])
    rt = []
    slid = rng.random((J, I)) < 0.08          # landslide blocks persist from period 12
    for t in range(T):
        for b in bases:
            for i, u in enumerate(units.district):
                base = float(tau0[(tau0.origin == b) & (tau0.district == u)].tau0_hours.iloc[0])
                j = bases.index(b)
                flooded = true[t, i] >= 3
                slide = bool(slid[j, i]) and t >= 12
                rt.append({"date": dates[t], "origin": b, "district": u, "mode": "road",
                           "tt_hours": np.inf if (flooded or slide) else base * rng.uniform(1, 1.5)})
                rt.append({"date": dates[t], "origin": b, "district": u, "mode": "boat",
                           "tt_hours": np.inf if slide else base * (2.0 if flooded else 1.0)})
    route_times = pd.DataFrame(rt)
    stocks = pd.DataFrame([{"base": b, "resource": r, "count": c}
                           for b in bases for r, c in (("team", 10), ("boat", 12), ("medical", 4))])
    return dict(forecasts=forecasts, units=units, exposure=exposure, tau0=tau0,
                route_times=route_times, stocks=stocks, calibration=cal)


In [ ]:
import importlib, cgrr, routes, synthetic
for m in (cgrr, routes, synthetic): importlib.reload(m)
cfg = cgrr.Config(**PARAMS)
import pandas as pd, numpy as np, json, hashlib, time, platform

## 3. Inputs

For each event you need the files below in `DATA`. Files marked ★ come from notebook 01.

| File | Columns | Source |
|---|---|---|
| ★ `kerala_flood_exposure_all.csv` | event, date, district, exposed_pop, total_pop | notebook 01 |
| ★ `kerala_travel_time_tau0.csv` | origin, district, tau0_hours | notebook 01 (with your BASES) |
| `forecasts_{event}.csv` | period, datetime, district, p_low, p_medium, p_high, p_severe, [pred_depth_m], [true_class] | your Paper 3 multi-hazard model, re-run every 6 h over the event |
| `calibration.csv` | period, district, p_low…p_severe, true_class, [exposed_frac] | the same model on pre-event monsoon periods (never the test events) |
| `stocks_{event}.csv` | base, resource (team/boat/medical), count | NDRF records (`data/external/` in the repository) |
| `route_times_{event}.csv` | date, origin, district, mode (road/boat), tt_hours | built in step 3b |
| `vulnerability.csv` (optional) | district, w, v | your vulnerability index; default 1 |

`true_class` is the observed risk class. If it's missing, the notebook derives it from the observed exposed fraction (thresholds 1 %, 5 %, 15 %), and you must state that in the paper.

### 3b. Realized route times from flooded roads and landslides
Run this once per event. Road resources can't use flooded or landslide-blocked segments; boats can't use landslide-blocked segments and move at a third of the speed on flooded ones.

**Landslide dates:** the 2018 inventory (Hao et al. 2020) has no reliable date for every point, so each landslide is treated as blocking from `LANDSLIDE_ACTIVE_FROM`. State this in the paper. For 2019, make `landslides_2019.csv` (lat, lon, date) from the sources in `data/SOURCES.md`.

In [ ]:
BUILD_ROUTES = not SMOKE_TEST
LANDSLIDE_FILES = {'2018': DATA + '/landslides_2018/landslides_2018.shp',   # Hao et al. 2020 shapefile
                   '2019': DATA + '/landslides_2019.csv'}                   # lat, lon, date
LANDSLIDE_ACTIVE_FROM = {'2018': '2018-08-15', '2019': '2019-08-08'}
BASES = {}   # same dict as in notebook 01: 'Base name': (lat, lon); districts are added automatically

if BUILD_ROUTES:
    import osmnx as ox, geopandas as gpd
    G = ox.load_graphml(DATA + '/kerala_roads.graphml')
    edges = gpd.read_file(DATA + '/kerala_roads.gpkg')
    dist = gpd.read_file(DATA + '/kerala_districts.geojson').to_crs(4326)
    pts = dist.geometry.representative_point()
    dests = dict(zip(dist['district'], ox.distance.nearest_nodes(G, pts.x, pts.y)))
    origins = dict(dests)
    for b, (lat, lon) in BASES.items():
        origins[b] = ox.distance.nearest_nodes(G, lon, lat)
    flooded = pd.read_csv(DATA + '/kerala_flooded_roads.csv')
    for ev_name in EVENTS:
        fl = flooded[flooded['date'].str.startswith(ev_name)]
        dates = sorted(fl['date'].unique())
        f_by = {d: set(g['edge_id']) for d, g in fl.groupby('date')}
        lf = LANDSLIDE_FILES[ev_name]
        if lf.endswith('.csv'):
            l = pd.read_csv(lf); lgdf = gpd.GeoDataFrame(l, geometry=gpd.points_from_xy(l.lon, l.lat), crs=4326)
        else:
            lgdf = gpd.read_file(lf)
        blocked = routes.blocked_edges_from_landslides(edges, lgdf, buffer_m=50)
        b_by = {d: (blocked if d >= LANDSLIDE_ACTIVE_FROM[ev_name] else set()) for d in dates}
        rt = routes.route_times(G, edges[['edge_id', 'u', 'v', 'key']], origins, dests,
                                dates, f_by, b_by)
        rt.to_csv(f'{DATA}/route_times_{ev_name}.csv', index=False)
        print(ev_name, len(dates), 'dates,', len(blocked), 'landslide-blocked segments')

### 3c. Load events

In [ ]:
def file_hash(p):
    return hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16] if os.path.exists(p) else 'missing'

events, cals, input_hashes = {}, {}, {}
if SMOKE_TEST:
    for k, ev_name in enumerate(EVENTS):
        d = synthetic.make(seed=k)
        events[ev_name] = cgrr.load_event('SMOKE-' + ev_name, d['forecasts'], d['units'], d['exposure'],
                                          d['tau0'], d['route_times'], d['stocks'], cfg)
        cals[ev_name] = d['calibration']
else:
    exposure = pd.read_csv(DATA + '/kerala_flood_exposure_all.csv')
    tau0 = pd.read_csv(DATA + '/kerala_travel_time_tau0.csv')
    cal = pd.read_csv(DATA + '/calibration.csv')
    vul = pd.read_csv(DATA + '/vulnerability.csv') if os.path.exists(DATA + '/vulnerability.csv') else None
    for ev_name in EVENTS:
        ex = exposure[exposure['event'].astype(str) == ev_name]
        units = ex.groupby('district', as_index=False)['total_pop'].max()
        if vul is not None:
            units = units.merge(vul, on='district', how='left').fillna({'w': 1.0, 'v': 1.0})
        files = {k: f'{DATA}/{k}_{ev_name}.csv' for k in ('forecasts', 'stocks', 'route_times')}
        for k, p in files.items():
            assert os.path.exists(p), f'Missing input: {p}'
        events[ev_name] = cgrr.load_event(ev_name, pd.read_csv(files['forecasts']), units, ex, tau0,
                                          pd.read_csv(files['route_times']), pd.read_csv(files['stocks']), cfg)
        cals[ev_name] = cal
        input_hashes[ev_name] = {k: file_hash(p) for k, p in files.items()}
    input_hashes['shared'] = {f: file_hash(f'{DATA}/{f}') for f in
                              ('kerala_flood_exposure_all.csv', 'kerala_travel_time_tau0.csv', 'calibration.csv')}
for k, ev in events.items():
    print(k, f'units={ev.I} bases={ev.J} periods={ev.T} ({ev.periods[0]} to {ev.periods[-1]})',
          'stocks:', dict(zip(ev.res_names, ev.stocks.sum(1))))

## 4. Main comparison and ablations (Table 4, RQ1–RQ3)

In [ ]:
LOG = []
def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S ') + str(msg); print(line); LOG.append(line)

log(f'START  SMOKE_TEST={SMOKE_TEST}  python={platform.python_version()}  '
    f'numpy={np.__version__}  pandas={pd.__version__}  pulp={cgrr.pulp.__version__}')
log('PARAMS ' + json.dumps({k: v for k, v in PARAMS.items()}, default=str))
main = {}
for ev_name, ev in events.items():
    t0 = time.time()
    main[ev_name] = cgrr.run_event(ev, cals[ev_name], cfg, log=log)
    log(f'[{ev_name}] main done in {time.time() - t0:.0f} s')
    main[ev_name]['summary'].to_csv(f'{OUT}/table4_{ev_name}.csv', index=False)
    main[ev_name]['tests'].to_csv(f'{OUT}/tests_{ev_name}.csv', index=False)
    for m, df in main[ev_name]['results'].items():
        df.to_csv(f'{OUT}/runs_{ev_name}_{m}.csv', index=False)
    plans = main[ev_name]['plans']['CGRR']
    pd.DataFrame({'period': range(ev.T), 'datetime': ev.periods, 'replanned': plans.replans,
                  'reason': plans.reason}).to_csv(f'{OUT}/trigger_log_{ev_name}.csv', index=False)

In [ ]:
COLS = ['worst_unit_weighted_unmet', 'total_unmet_people', 'mean_response_h', 'p95_response_h',
        'share_within_6h_pct', 'coverage_pct', 'missed_high_risk', 'relocations', 'replans', 'failed_dispatches']
for ev_name in events:
    print(f'===== {ev_name}: medians over {cfg.n_runs} runs =====')
    display(main[ev_name]['summary'].set_index('method')[COLS].round(2))
    print('Calibration:', {k: (np.round(v, 4) if isinstance(v, (float, np.ndarray)) else v)
                           for k, v in main[ev_name]['info'].items()})
    print('LaTeX rows for Table 4:')
    print(cgrr.latex_main_table(main[ev_name]['summary'], ['B1', 'B2', 'B3', 'B4', 'B5', 'CGRR']))

### Significance (CGRR vs each method, Holm-corrected)
`median_diff_ref_minus_method` < 0 means CGRR is lower. That's better for unmet demand and response time, and worse for coverage.

In [ ]:
for ev_name in events:
    t = main[ev_name]['tests']
    key = t[t['metric'].isin(['worst_unit_weighted_unmet', 'total_unmet_people', 'coverage_pct',
                              'mean_response_h', 'relocations', 'failed_dispatches'])]
    print('=====', ev_name); display(key.pivot(index='method', columns='metric', values='p_holm').round(4))

## 5. Stress scenarios (Table 5)

In [ ]:
stress = {}
for ev_name, ev in events.items():
    t0 = time.time()
    stress[ev_name], _ = cgrr.run_stress(ev, cals[ev_name], cfg, main[ev_name], log=log)
    stress[ev_name].to_csv(f'{OUT}/table5_stress_{ev_name}.csv', index=False)
    log(f'[{ev_name}] stress done in {time.time() - t0:.0f} s')
    display(stress[ev_name].round(2))

## 6. Sensitivity to α and η

In [ ]:
sens = {}
for ev_name, ev in events.items():
    t0 = time.time()
    sens[ev_name] = cgrr.run_sensitivity(ev, cals[ev_name], cfg, log=log)
    sens[ev_name].to_csv(f'{OUT}/sensitivity_{ev_name}.csv', index=False)
    log(f'[{ev_name}] sensitivity done in {time.time() - t0:.0f} s')
    display(sens[ev_name].round(3))

## 7. Figures (sans-serif text, patterns plus colour; save as 600 dpi TIFF for submission)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
for ev_name, ev in events.items():
    s = sens[ev_name]
    fig, ax = plt.subplots(1, 2, figsize=(6.85, 2.6))   # 174 mm wide
    for e, mk in zip(sorted(s.eta.unique()), 'osd^v'):
        g = s[s.eta == e]
        ax[0].plot(g.alpha, g.total_weighted_unmet, marker=mk, label=f'η={e}')
    ax[0].set_xlabel('α'); ax[0].set_ylabel('Total weighted unmet demand'); ax[0].legend(fontsize=7)
    g = s[s.eta == cfg.eta]
    ax[1].plot(1 - g.alpha, g.set_coverage, 'o-', label='empirical')
    ax[1].plot([0.6, 1], [0.6, 1], 'k--', lw=0.8, label='target')
    ax[1].set_xlabel('Target coverage 1−α'); ax[1].set_ylabel('Empirical coverage'); ax[1].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f'{OUT}/Fig_alpha_{ev_name}.tiff', dpi=600); plt.show()

    tl = pd.read_csv(f'{OUT}/trigger_log_{ev_name}.csv', parse_dates=['datetime'])
    dhat = (ev.v * ev.P) * (ev.probs @ main[ev_name]['delta'])
    fig, ax = plt.subplots(figsize=(6.85, 2.2))
    ax.plot(tl.datetime, (ev.w * dhat).sum(1), 'k-', lw=1, label='forecast weighted demand')
    for r, mk in (('shift', '^'), ('escalation', 's'), ('route', 'x')):
        m = tl.reason.fillna('').str.contains(r)
        ax.scatter(tl.datetime[m], (ev.w * dhat).sum(1)[m], marker=mk, label=f'trigger: {r}', zorder=3)
    ax.set_ylabel('Weighted demand'); ax.legend(fontsize=7, ncol=4); fig.autofmt_xdate()
    fig.tight_layout(); fig.savefig(f'{OUT}/Fig_trigger_{ev_name}.tiff', dpi=600); plt.show()

## 8. Execution log (proof of execution) and numbers for the paper

In [ ]:
log('END')
with open(f'{OUT}/run_log.txt', 'w') as f:
    f.write('\n'.join(LOG) + '\n')
    f.write('INPUT_SHA256_16 ' + json.dumps(input_hashes) + '\n')
print(open(f'{OUT}/run_log.txt').read()[-3000:])

if SMOKE_TEST:
    print('\n*** SMOKE TEST: these numbers are random and must NOT be used in the paper. ***')
else:
    for ev_name in events:
        s = main[ev_name]['summary'].set_index('method')
        base = [m for m in ['B1', 'B2', 'B3', 'B4', 'B5']]
        best = min(base, key=lambda m: s.loc[m, 'total_unmet_people'])
        c, b = s.loc['CGRR'], s.loc[best]
        info = main[ev_name]['info']
        print(f'--- {ev_name} (for Abstract / Sect. 8 / Conclusions) ---')
        print(f'Conformal coverage {info["set_coverage"]:.3f} at target {1 - cfg.alpha:.2f}; mean set size {info["mean_set_size"]:.2f}; Gamma={info["Gamma"]}')
        print(f'Best baseline by total unmet demand: {best}')
        print(f'CGRR vs {best}: unmet {100 * (c.total_unmet_people / b.total_unmet_people - 1):+.1f} %, '
              f'worst-unit {100 * (c.worst_unit_weighted_unmet / b.worst_unit_weighted_unmet - 1):+.1f} %, '
              f'coverage {c.coverage_pct - b.coverage_pct:+.1f} pp, mean response {c.mean_response_h - b.mean_response_h:+.2f} h, '
              f'relocations {c.relocations:.0f} vs {b.relocations:.0f}')
        print(f'RQ2: CGRR {s.loc["CGRR","replans"]:.0f} re-plans / {s.loc["CGRR","relocations"]:.0f} relocations vs '
              f'A2a {s.loc["A2a","replans"]:.0f} / {s.loc["A2a","relocations"]:.0f}; coverage {s.loc["CGRR","coverage_pct"]:.1f} vs {s.loc["A2a","coverage_pct"]:.1f} %')
        print(f'RQ3: failed dispatches CGRR {s.loc["CGRR","failed_dispatches"]:.0f} vs A3 {s.loc["A3","failed_dispatches"]:.0f}')
    print('\nReport results honestly, including where CGRR does not beat a baseline.')